In [1]:
import torch
import transformers, datasets, tokenizers, peft
print("transformers:", transformers.__version__)
print("datasets    :", datasets.__version__)
print("tokenizers  :", tokenizers.__version__)
print("peft        :", peft.__version__)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device      :", device)

transformers: 5.17.0
datasets    : 5.0.1
tokenizers  : 0.23.2
peft        : 0.21.1
device      : cpu


In [2]:

from transformers import pipeline

sentiment = pipeline("sentiment-analysis")
print(sentiment("I love this course!"))
# [{'label': 'POSITIVE', 'score': 0.9998}]

# Common task pipelines:
pipeline("text-generation")(  "Once upon a time", max_new_tokens=30)
pipeline("summarization")(    long_text, max_length=60, min_length=20)
pipeline("translation_en_to_fr")("Hello, how are you?")
pipeline("zero-shot-classification")("This is a tutorial about AI", candidate_labels=["tech","sports","cooking"])
pipeline("question-answering")(question="Who built AlfredX?", context="Naved built AlfredX with 3 teammates.")
pipeline("fill-mask")("Paris is the [MASK] of France.")

# Pick a specific model instead of the default:
pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

# Run on GPU:
pipeline("sentiment-analysis", device=0)   # device=0 -> first GPU, device=-1 (default) -> CPU

print("reference code above — run on a machine with internet")

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[transformers] No model was supplied, defaulted to HuggingFaceTB/SmolLM3-3B and revision a07cc9a.
Using a pipeline without specifying a model name and revision in production is not recommended.


[{'label': 'POSITIVE', 'score': 0.9998835325241089}]


Loading weights:   0%|          | 0/326 [00:01<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=30) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


KeyError: "Unknown task summarization, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection']"

In [3]:
# --- Example 1: train a Byte-Pair Encoding tokenizer on sample text ---
from tokenizers import Tokenizer, models, trainers, pre_tokenizers

corpus = [
    "AlfredX is an AI voice assistant built with PyQt5 and FastAPI.",
    "The assistant uses the Groq API and Ollama for language understanding.",
    "It supports over one hundred voice and text commands.",
    "Naved trained a tokenizer to understand subword tokenization deeply.",
    "Transformers use self-attention to process sequences in parallel.",
    "Fine-tuning adapts a pretrained model to a new, smaller dataset.",
]

tok = Tokenizer(models.BPE(unk_token="[UNK]"))
tok.pre_tokenizer = pre_tokenizers.Whitespace()
trainer = trainers.BpeTrainer(vocab_size=300, special_tokens=["[UNK]", "[PAD]", "[CLS]", "[SEP]", "[MASK]"])
tok.train_from_iterator(corpus, trainer)

print("Vocab size:", tok.get_vocab_size())
enc = tok.encode("AlfredX understands tokenization.")
print("Tokens:", enc.tokens)
print("IDs   :", enc.ids)

Vocab size: 208
Tokens: ['AlfredX', 'understand', 's', 'tokenization', '.']
IDs   : [172, 92, 36, 170, 7]


In [4]:
# --- Example 2: wrap it as a real HF PreTrainedTokenizerFast (this IS what AutoTokenizer returns) ---
from transformers import PreTrainedTokenizerFast

hf_tok = PreTrainedTokenizerFast(
    tokenizer_object=tok,
    unk_token="[UNK]", pad_token="[PAD]", cls_token="[CLS]", sep_token="[SEP]", mask_token="[MASK]",
)

out = hf_tok("AlfredX is an AI voice assistant.", padding="max_length", max_length=16, truncation=True)
print("input_ids     :", out["input_ids"])
print("attention_mask:", out["attention_mask"])   # 1 = real token, 0 = padding -- tells the model what to ignore
print("decoded       :", hf_tok.decode(out["input_ids"], skip_special_tokens=True))

input_ids     : [172, 124, 44, 94, 81, 93, 7, 1, 1, 1, 1, 1, 1, 1, 1, 1]
attention_mask: [1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]
decoded       : AlfredX is an AI voice assistant .


In [5]:
batch = ["AlfredX handles reminders.", "Transformers use self-attention to process sequences in parallel efficiently."]
enc = hf_tok(batch, padding=True, truncation=True, max_length=12, return_tensors="pt")
print("input_ids shape:", enc["input_ids"].shape)   # both padded to the same length
for ids, mask in zip(enc["input_ids"], enc["attention_mask"]):
    print(ids.tolist(), "| mask:", mask.tolist())

input_ids shape: torch.Size([2, 12])
[172, 26, 47, 29, 23, 36, 35, 23, 30, 46, 22, 45] | mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
[204, 80, 149, 6, 198, 51, 200, 197, 46, 192, 23, 24] | mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [6]:
# --- Example 1: build a real (tiny) BERT architecture offline via AutoConfig + from_config ---
from transformers import AutoConfig, AutoModel, AutoModelForSequenceClassification

config = AutoConfig.for_model(
    "bert", vocab_size=hf_tok.vocab_size, hidden_size=64, num_hidden_layers=2,
    num_attention_heads=4, intermediate_size=128, max_position_embeddings=32,
)
model = AutoModel.from_config(config)
print(model.config)
print("Total params:", sum(p.numel() for p in model.parameters()))

BertConfig {
  "add_cross_attention": false,
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": null,
  "classifier_dropout": null,
  "eos_token_id": null,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 64,
  "initializer_range": 0.02,
  "intermediate_size": 128,
  "is_decoder": false,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 32,
  "model_type": "bert",
  "num_attention_heads": 4,
  "num_hidden_layers": 2,
  "pad_token_id": 0,
  "tie_word_embeddings": true,
  "transformers_version": "5.17.0",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 208
}

Total params: 86720


In [7]:
# --- Example 2: run text through it and inspect the outputs ---
enc = hf_tok("AlfredX is an AI voice assistant.", return_tensors="pt", padding="max_length", max_length=16, truncation=True)
with torch.no_grad():
    out = model(**enc)

print("last_hidden_state:", out.last_hidden_state.shape)   # (batch, seq_len, hidden_size) -- one vector per token
print("pooler_output     :", out.pooler_output.shape)       # (batch, hidden_size) -- one vector for the whole sequence (from [CLS])

last_hidden_state: torch.Size([1, 16, 64])
pooler_output     : torch.Size([1, 64])


In [8]:
# --- Example 3: a classification head on top (this is what you fine-tune for sentiment, etc.) ---
config.num_labels = 2                     # set on the config, not as a from_config() kwarg
clf_model = AutoModelForSequenceClassification.from_config(config)
with torch.no_grad():
    logits = clf_model(**enc).logits
print("logits:", logits)                          # raw scores, one per class
print("probs :", logits.softmax(dim=-1))           # turn into probabilities

logits: tensor([[-0.0061, -0.0064]])
probs : tensor([[0.5001, 0.4999]])


In [9]:
# --- Example 1: build a Dataset in memory (offline) ---
from datasets import Dataset, DatasetDict

texts = [
    "AlfredX handles reminders and timers well.", "The rate-limit retry logic keeps the app stable.",
    "The bug caused the app to crash randomly.", "Broken context handling frustrated users.",
    "The bilingual interface is a nice touch.", "Streaming responses feel fast and smooth.",
    "The duplicate class definition caused confusing errors.", "Missing conversation history was a major bug.",
    "The JARVIS-style HUD looks great on screen.", "The WebSocket companion app works reliably.",
]
labels = [1, 1, 0, 0, 1, 1, 0, 0, 1, 1]   # 1 = positive, 0 = negative

ds = Dataset.from_dict({"text": texts, "label": labels})
print(ds)
print(ds[0])
split = ds.train_test_split(test_size=0.3, seed=42)
print(split)

Dataset({
    features: ['text', 'label'],
    num_rows: 10
})
{'text': 'AlfredX handles reminders and timers well.', 'label': 1}
DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 7
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 3
    })
})


In [10]:
# --- Example 2: .map() to tokenize the whole dataset at once ---
def tokenize_fn(batch):
    return hf_tok(batch["text"], padding="max_length", truncation=True, max_length=20)

tokenized = split.map(tokenize_fn, batched=True)
print(tokenized["train"][0].keys())
print(tokenized["train"][0]["input_ids"])

Map:   0%|          | 0/7 [00:00<?, ? examples/s]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

dict_keys(['text', 'label', 'input_ids', 'attention_mask'])
[0, 27, 60, 84, 21, 58, 39, 45, 36, 156, 26, 68, 54, 42, 40, 19, 36, 19, 73, 0]


In [11]:
# --- Example 3: filter, sort, and set the output format for PyTorch ---
short_only = ds.filter(lambda row: len(row["text"]) < 40)
print("Rows with short text:", len(short_only))

tokenized.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
print(type(tokenized["train"][0]["input_ids"]))   # now a torch.Tensor, ready for the model/Trainer

Filter:   0%|          | 0/10 [00:00<?, ? examples/s]

Rows with short text: 0
<class 'torch.Tensor'>


In [12]:
from transformers import TrainingArguments, Trainer
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {"accuracy": (preds == labels).mean()}

args = TrainingArguments(
    output_dir="hf_out",
    num_train_epochs=8,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    eval_strategy="epoch",
    save_strategy="no",
    logging_strategy="epoch",
    learning_rate=5e-4,
    report_to=[],             # disable wandb/etc logging integrations
)

trainer = Trainer(
    model=clf_model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    compute_metrics=compute_metrics,
)
trainer.train()

c:\Users\fragr\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy
1,0.694929,0.697829,0.666667
2,0.677223,0.705471,0.666667
3,0.677078,0.712800,0.666667
4,0.665828,0.719225,0.000000
5,0.665128,0.724862,0.000000
6,0.661260,0.729213,0.000000
7,0.659528,0.732078,0.000000
8,0.659621,0.733541,0.000000


TrainOutput(global_step=16, training_loss=0.6700743585824966, metrics={'train_runtime': 0.9864, 'train_samples_per_second': 56.774, 'train_steps_per_second': 16.221, 'total_flos': 479552640.0, 'train_loss': 0.6700743585824966, 'epoch': 8.0})

In [13]:
metrics = trainer.evaluate()
print(metrics)

preds = trainer.predict(tokenized["test"])
pred_labels = np.argmax(preds.predictions, axis=1)
print("Predicted:", pred_labels.tolist())
print("True     :", preds.label_ids.tolist())

Training Loss,Validation Loss,Epoch,Accuracy
0.659621,0.733541,8,0.000000


{'eval_loss': 0.733540952205658, 'eval_accuracy': 0.0}


Predicted: [0, 1, 0]
True     : [1, 0, 1]


In [16]:

import evaluate
acc = evaluate.load("accuracy")
f1  = evaluate.load("f1")
print(acc.compute(predictions=pred_labels, references=preds.label_ids))
print(f1.compute(predictions=pred_labels, references=preds.label_ids))

combo = evaluate.combine(["accuracy", "f1", "precision", "recall"])
print(combo.compute(predictions=pred_labels, references=preds.label_ids))


# Offline equivalent (same metrics, no download) -- what we'll actually run here:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

print("accuracy :", accuracy_score(preds.label_ids, pred_labels))
print("f1       :", f1_score(preds.label_ids, pred_labels))
print("precision:", precision_score(preds.label_ids, pred_labels))
print("recall   :", recall_score(preds.label_ids, pred_labels))

{'accuracy': 0.0}
{'f1': 0.0}


{'accuracy': 0.0, 'f1': 0.0, 'precision': 0.0, 'recall': 0.0}
accuracy : 0.0
f1       : 0.0
precision: 0.0
recall   : 0.0
